# CBT Knowledge RAG v4 — Full-System Comparison

This notebook analyses the fresh 16-case comparison between no RAG and the original best-performing seven-source multilingual hybrid + cross-encoder RAG. It reads the committed CSV outputs and does not call the generation API.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest, wilcoxon

RESULTS = Path('results')
NO = 'no_rag'
RAG = 'full_hybrid_rerank_rag'
detail = pd.read_csv(RESULTS / 'full_rag_dialogue_v4_results.csv')
scores = pd.read_csv(RESULTS / 'full_rag_dialogue_v4_model_assisted_scores.csv')
summary = pd.read_csv(RESULTS / 'full_rag_dialogue_v4_summary.csv')
audit = pd.read_csv(RESULTS / 'full_rag_dialogue_v4_retrieval_audit.csv')
retrieval = pd.read_csv(RESULTS / 'retrieval_summary.csv')
summary

## 1. Verify that the original best retriever was restored

The response comparison should only be interpreted if the full retriever reproduces its earlier retrieval metrics.

In [ ]:
retrieval[['method', 'recall_at_5', 'recall_at_10', 'mrr_at_10',
           'context_precision_at_5', 'safety_recall_at_5']]

Expected `hybrid_rerank` values: Recall@5 0.80, Recall@10 0.84, MRR@10 about 0.583, context precision@5 0.44, and safety Recall@5 1.00.

## 2. Headline response results

In [ ]:
summary[['arm', 'n', 'mean_model_assisted_score_12', 'pairwise_wins', 'ties',
         'citation_rate', 'truncation_rate', 'median_latency_ms']]

In [ ]:
score_cols = [f'{NO}_total', f'{RAG}_total']
plot_df = scores.set_index('case_id')[score_cols].rename(
    columns={f'{NO}_total': 'No RAG', f'{RAG}_total': 'Full RAG'})
ax = plot_df.plot(kind='bar', figsize=(12, 4), ylim=(0, 12.5), width=0.8)
ax.set_ylabel('Model-assisted score (/12)')
ax.set_title('Paired case-level CBT response scores')
ax.legend(frameon=False)
plt.tight_layout();

## 3. Dimension-level comparison

In [ ]:
dims = ['cbt_accuracy', 'collaboration_empathy', 'guided_discovery',
        'actionability', 'context_fit', 'safety_scope']
dimension_rows = []
for dim in dims:
    no_mean = scores[f'{NO}_{dim}'].mean()
    rag_mean = scores[f'{RAG}_{dim}'].mean()
    dimension_rows.append({'dimension': dim, 'no_rag': no_mean,
                           'full_rag': rag_mean, 'difference': rag_mean-no_mean})
dimension_summary = pd.DataFrame(dimension_rows)
dimension_summary

## 4. Paired uncertainty checks

These are exploratory development-set statistics. They do not correct for model-judge bias, ceiling effects, or the researcher-authored scenario set.

In [ ]:
diff = scores[f'{RAG}_total'] - scores[f'{NO}_total']
rng = np.random.default_rng(20260828)
bootstrap_means = np.array([rng.choice(diff.to_numpy(), len(diff), replace=True).mean()
                            for _ in range(100_000)])
nonzero = diff[diff != 0]
pd.Series({
    'paired_mean_difference': diff.mean(),
    'bootstrap_95_low': np.quantile(bootstrap_means, 0.025),
    'bootstrap_95_high': np.quantile(bootstrap_means, 0.975),
    'rag_better_non_ties': int((nonzero > 0).sum()),
    'no_rag_better_non_ties': int((nonzero < 0).sum()),
    'sign_test_p_two_sided': binomtest(int((nonzero > 0).sum()), len(nonzero), 0.5).pvalue,
    'wilcoxon_p_two_sided': wilcoxon(diff, zero_method='wilcox').pvalue,
})

In [ ]:
case_view = scores[['case_id', 'category', f'{NO}_total', f'{RAG}_total', 'winner', 'reason']].copy()
case_view['difference'] = diff
case_view.sort_values(['difference', 'case_id'], ascending=[False, True])

## 5. Retrieval audit

This table makes it possible to inspect which sources actually entered the prompt. A retrieved passage can be traceable but still irrelevant.

In [ ]:
pd.crosstab(audit['source_id'], audit['rank'])

In [ ]:
audit[audit['rank'] == 1][['case_id', 'category', 'source_id', 'citation', 'topics']]

## Interpretation

Full RAG produced a small positive development-set difference (+0.3125/12; six wins, two losses, eight ties), mainly in guided discovery. The bootstrap interval crossed zero and the baseline was near ceiling, so the result is preliminary. This supports freezing the full RAG as a common knowledge component for later memory ablations; it does not establish clinical effectiveness.